# Native CLM v0 — M3R Address Diagnostic

Checkpoint-only diagnostic over the already-consumed canonical M3R lineage checkpoints. This notebook performs **no Native CLM training, no Cell updates, no routing updates, no certificate updates, and no growth**.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/native-clm-v0-m3r-address-diagnostic'
REPO = Path('/kaggle/working/mini-cells')
DATA = Path('/kaggle/working/native-clm-m3r-address-data')
CHECKPOINTS = Path('/kaggle/working/native-clm-m3r-address-checkpoints')
OUT = REPO / 'artifacts/experiments/native-clm-v0-m3r-address-diagnostic'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', REPO])
else:
    os.chdir(REPO)
    run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
    run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'])
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing Kaggle Secret: HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing Kaggle Secret: GITHUB_TOKEN'
print('Secrets loaded. This stage requires HF read access and GitHub write access.')

In [ ]:
import torch

print('torch:', torch.__version__)
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Canonical diagnostic expects two GPUs.'

## Recover the exact M3R data snapshot

The script uses the exact HF revisions stored in the canonical M3R data manifest and refuses to continue unless all seven reconstructed corpus files match the published byte counts and SHA-256 values.

In [ ]:
run([
    sys.executable,
    'scripts/research/prepare_native_clm_v0_m3r_address_data.py',
    '--output-dir', DATA,
])

## Fetch the exact published M3R lineage checkpoints

Only `lineage_growth` checkpoints at the frozen HF revision are downloaded. Every file is verified against the M3R model-artifact manifest.

In [ ]:
run([
    sys.executable,
    'scripts/research/fetch_native_clm_v0_m3r_address_checkpoints.py',
    '--output-dir', CHECKPOINTS,
])

## Run checkpoint-only address probes

GPU 0 and GPU 1 process different frozen M3R checkpoints concurrently. Domain labels are visible only to the offline diagnostic probes; the Native CLM learner is never updated.

In [ ]:
run([
    sys.executable,
    'scripts/research/run_native_clm_v0_m3r_address_diagnostic.py',
    '--data-dir', DATA,
    '--checkpoint-dir', CHECKPOINTS,
    '--output-dir', OUT,
    '--devices', 'cuda:0,cuda:1',
])

In [ ]:
result = json.loads((OUT / 'diagnostic-result.json').read_text())
print('classification:', result['classification'])
print('valid edges:', result['valid_edge_count'], '/', result['edge_count'])
print('current cosine median AUC:', result['current_cosine']['median_auc'])
for name, metrics in result['features'].items():
    print(name, metrics)
print('\n', result['interpretation'])

## Publish lightweight diagnostic evidence

No checkpoint is created or uploaded by this stage. Only JSON/CSV/Markdown diagnostic evidence is committed to the research branch.

In [ ]:
run([
    sys.executable,
    'scripts/research/publish_native_clm_v0_m3r_address_diagnostic.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
])
print('Published M3R address diagnostic:', result['classification'])